# MVTec AD — Data Exploration
Sanity check for the dataloader. Run after `python scripts/download_mvtec.py`.

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import torch
from pathlib import Path
from data.mvtec import MVTecDataset
from utils.seed import seed_everything

seed_everything(42)

In [ ]:
DATA_ROOT = Path('../data/mvtec')
CATEGORY  = 'bottle'

train_ds = MVTecDataset(DATA_ROOT, CATEGORY, 'train')
val_ds   = MVTecDataset(DATA_ROOT, CATEGORY, 'val')
test_ds  = MVTecDataset(DATA_ROOT, CATEGORY, 'test')

print(f'Train : {len(train_ds)} samples')
print(f'Val   : {len(val_ds)} samples')
print(f'Test  : {len(test_ds)} samples')

In [ ]:
# visualise 4 training samples (all normal)
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
fig.suptitle(f'Training samples — {CATEGORY} (all normal)', fontsize=13)

# ImageNet de-normalisation for display
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

def denorm(t):
    return (t * std + mean).permute(1, 2, 0).clamp(0, 1).numpy()

for ax, idx in zip(axes, [0, 1, 2, 3]):
    item = train_ds[idx]
    ax.imshow(denorm(item['image']))
    ax.set_title(f'label={item["label"]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# find an anomaly sample in test and show image + mask side by side
anomaly_items = [test_ds[i] for i in range(len(test_ds)) if test_ds[i]['label'] == 1]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle(f'Test anomaly samples — {CATEGORY}', fontsize=13)

for col, item in enumerate(anomaly_items[:4]):
    axes[0, col].imshow(denorm(item['image']))
    axes[0, col].set_title('image')
    axes[0, col].axis('off')
    axes[1, col].imshow(item['mask'].squeeze(), cmap='hot')
    axes[1, col].set_title('GT mask')
    axes[1, col].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# class distribution in test set
labels = [test_ds[i]['label'] for i in range(len(test_ds))]
n_normal  = labels.count(0)
n_anomaly = labels.count(1)
print(f'Test set — normal: {n_normal}  anomaly: {n_anomaly}')
print(f'Anomaly rate: {n_anomaly / len(labels):.1%}')